# Middlewares

Middlewares are used to get more stricter control over the agents behaviour and the actions,
- it is used to decide the tool calls,
- retries, fallbacks and all.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

## Summarization Middleware
in this middleware the llm summarizes the chat when the limit is exceeded

###  Checkpointer
checkpointer is used to store the states of the llm suppose a task 
- User:
    - Plan a trip to Goa
- workflow will be 
    - Step 1 → Search hotels
    - Step 2 → Search flights
    - Step 3 → Create itinerary

if there is any failure in the agent, the states may be lost if there is no checkpointer,
checkpointer saves every state of conversation and the workflow step by step and


### InMemorySaver
 is the simplest checkpointer, it stores the messages in the RAM and helps llm to retrieve if anything is lost



 ### Trigger
 it contains arguements like the list/container , length . when len exceeds middleware is triggered



 ### Keep 
 keep keeps the recent top 4 messages

### Configurable
configurable gives the keys to the memory to save the chats into a specific thread id, 
InMemorySaver saves the content in dictionary format
memory:{thread1:{}
        thread2:{}}
configurable allows us to give that key to memory to save the context 

while saving the langgraph checks threadid : [configurable][thread_id]
that's why we follow the format 

for multiple threads we do multple configs like 
- config1={'configurable':{"thread_id":"test_1"}}
- config2={'configurable':{"thread_id":"test_2"}}          

### Summarization using messages

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage

agent=create_agent(
    model='groq:qwen/qwen3-32b',
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='groq:qwen/qwen3-32b',
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [ ]:
config={'configurable':{"thread_id":"test_1"}}

In [ ]:
questions=['My name is Nitin and I am studying AIML. Remember this.',
'I am currently learning LangChain and LangGraph. Remember this too.',
'My favorite programming language is Java.',
'I want to build AI agents and RAG applications.',
'My favorite movie is Jersey.',
'What is my favorite movie and what technology am I learning?']

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(q)]},config)
    print("messages",response)
    print(len((response["messages"])))

### Summarization using tokens 

In [ ]:
agent1=create_agent(
    model='groq:qwen/qwen3-32b',
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='groq:qwen/qwen3-32b',
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)

config1={"configurable":{"thread_id":"test2"}}

In [ ]:
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages)

cities=["hyd","banglore","delhi","chennai"]

for city in cities:
    response=agent1.invoke({"messages":[f"tell about the famous things about the city {city}"]}, config1)
    tokens=count_tokens(response["messages"])
    print('tokens ',tokens)
    print(response["messages"])

 ### Summarization using Fractions
 Fractions-represents the amount of context of the llm used 

In [4]:
agent1=create_agent(
    model='groq:qwen/qwen3-32b',
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='groq:qwen/qwen3-32b',
            trigger=("fraction",0.005), ##0.5=640 words
            keep=("fraction",0.02)
        )
    ]
)

config2={"configurable":{"thread_id":"test3"}}

def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages)//4

cities=["hyd","banglore","delhi","chennai"]

for city in cities:
    response=agent1.invoke({"messages":[f"tell about the famous things about the city {city}"]}, config2)
    tokens=count_tokens(response["messages"])
    print('tokens ',tokens/12800)
    
    print(response["messages"])

tokens  0.15171875
[HumanMessage(content='tell about the famous things about the city hyd', additional_kwargs={}, response_metadata={}, id='03623325-36fb-4cbc-ba2f-c6e73b74887d'), AIMessage(content='<think>\nOkay, the user is asking about famous things in Hyderabad. Let me start by recalling what I know about the city. Hyderabad is a major city in India, known for its rich history and culture. It\'s the capital of Telangana and Andhra Pradesh, right? So I need to mention that.\n\nFirst, the historical aspects. The Charminar comes to mind immediately. It\'s a famous landmark. I should explain its significance, when it was built, maybe some architectural details. Then there\'s the Golconda Fort, another major site. I should talk about its history as a Qutb Shahi dynasty fort and its acoustics or the muses.\n\nTemples and other places of worship. The Mecca Masjid is a significant mosque. Also, the Salar Jung Museum has a lot of artifacts. The Birla Mandir is another religious site. Maybe 

## Human in the Loop Middleware
In this middleware the agents action until the human approves or edits or rejects the usage of the tool before they execute. 
- This is useful in high stake operations
- Also in long conversations

Implementation of Human in the Loop Middleware check it in 6.guardrails_learning